# Практическое долгосрочное задание 1
## 👄 Интерполяция формы губ по видео



## ЧАСТЬ 0. 🎯 Введение

Нам в руки попали **обрывки данных** — редкие снимки губ человека, произносящего звуки **восточноевропейского алфавита** 🕵️

Наш агент, внедрённый глубоко под прикрытием, снимал его **раз в секунду** — строго по **равномерной временно́й сетке**. Никаких отклонений, никаких дополнительных замеров. Только сухие факты: кадр, ещё кадр, и снова тишина.




Между этими кадрами — **пустота**. Звуки шли один за другим, но мы видим лишь **редкие вспышки мимики**: губы то округляются, то растягиваются, то сжимаются — и всё это **урывками**, без плавных переходов. Мимика кажется **рваной**, движения — **дёргаными**. Смотреть на такое больно 😖

---

### 🚀 Наша финальная задача

#### Научиться воспроизводить **гладкую анимацию** для… 🌀

…впрочем, узнаем **потом**. 😏

---

### 🎬 А пока — начнём с малого!

#### Попробуем восстановить хотя бы **отдельные звуки** 🗣️

<table>
<tr>
<td align="center">👄</td>
<td align="center">🎞️</td>
<td align="center">✨</td>
<td align="center">🔊</td>
</tr>
<tr>
<td align="center">губы</td>
<td align="center">кадры</td>
<td align="center">сгладим</td>
<td align="center">зазвучит!</td>
</tr>
</table>

---

> 💭 **Шаг за шагом:**
> 1. 📍 Соберём редкие кадры
> 2. 🧮 Построим интерполяцию
> 3. 🎥 Оживим анимацию
> 4. 🌍 А потом — уже что-то большее…

In [10]:
import sys
!{sys.executable} -m pip install matplotlib numpy ipython

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.3 MB 5.6 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/9.3 MB 5.9 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.3 MB 6.4 MB/s eta 0:00:01
   ---------------------- ----------------- 5.2/9.3 MB 6.5 MB/s eta 0:00:01
   ----------------------------- ---------- 6.8/9.3 MB 6.8 MB/s eta 0:00:01
   ----------------------------------- ---- 8.4/9.3 MB 6.8 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 6.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ----- ---------------------------------- 1.6/12.6 MB 8.4 MB/s eta 0:00:02
   ---------- ----------------------------- 3.4/12.6 MB 8.0 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/12.6 MB 7.9 MB/s eta 0:00:01
   -------------------- ----------


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
#Импорт нужных библиотек

from IPython.display import HTML

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy
import glob
from collections import defaultdict
from draw import *

import pickle

# Практическое долгосрочное задание 1
## 👄 Интерполяция формы губ по видео



## ЧАСТЬ 0. 🎯 Введение

Нам в руки попали **обрывки данных** — редкие снимки губ человека, произносящего звуки **восточноевропейского алфавита** 🕵️

Наш агент, внедрённый глубоко под прикрытием, снимал его **раз в секунду** — строго по **равномерной временно́й сетке**. Никаких отклонений, никаких дополнительных замеров. Только сухие факты: кадр, ещё кадр, и снова тишина.




Между этими кадрами — **пустота**. Звуки шли один за другим, но мы видим лишь **редкие вспышки мимики**: губы то округляются, то растягиваются, то сжимаются — и всё это **урывками**, без плавных переходов. Мимика кажется **рваной**, движения — **дёргаными**. Смотреть на такое больно 😖

---

### 🚀 Наша финальная задача

#### Научиться воспроизводить **гладкую анимацию** для… 🌀

…впрочем, узнаем **потом**. 😏

---

### 🎬 А пока — начнём с малого!

#### Попробуем восстановить хотя бы **отдельные звуки** 🗣️

<table>
<tr>
<td align="center">👄</td>
<td align="center">🎞️</td>
<td align="center">✨</td>
<td align="center">🔊</td>
</tr>
<tr>
<td align="center">губы</td>
<td align="center">кадры</td>
<td align="center">сгладим</td>
<td align="center">зазвучит!</td>
</tr>
</table>

---

> 💭 **Шаг за шагом:**
> 1. 📍 Соберём редкие кадры
> 2. 🧮 Построим интерполяцию
> 3. 🎥 Оживим анимацию
> 4. 🌍 А потом — уже что-то большее…

In [13]:
#Импорт нужных библиотек

from IPython.display import HTML

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy
import glob
from collections import defaultdict
from draw import *

import pickle

In [15]:
#Загрузка данных
with open('LipsArrayHalfTime (2).np' , 'rb') as f:
    LipsData = pickle.load(f)

### 📦 Структура данных: `LipsData`

`LipsData` — это **словарь**, ставящий в соответствие каждой букве её временну́ю траекторию движения губ:

$$
\text{LipsData} : \quad \text{буква} \;\longmapsto\; \mathbf{X} \in \mathbb{R}^{n \times 40 \times 2}
$$

---

#### 📐 Что означает каждый индекс

$$
\mathbf{X}[t,\, p,\, c], \qquad
\begin{aligned}
&t = 0, \dots, n-1 \\
&p = 0, \dots, 39 \\
&c \in \{0, 1\}
\end{aligned}
$$

| Индекс | Обозначение | Смысл |
|:---:|:---:|:---|
| $t$ | **время** | номер такта — дискретный отсчёт вдоль оси времени |
| $p$ | **точка** | 40 ключевых точек контура губ (20 внешних + 20 внутренних) |
| $c$ | **координата** | $c=0 \Rightarrow x$, $\;c=1 \Rightarrow y$ на плоскости |

---

#### 🕰️ О такте

> **Такт** — это условная единица времени.  
> Можно считать, что $1$ такт $\approx 0.1$ с, но эта привязка несущественна:  
> для интерполяции важна лишь **последовательность** отсчётов, а не их физическая длительность.

---

#### 🧠 Как это читать

Для фиксированной буквы $\ell$ имеем:

$$
\text{LipsData}[\ell] \;=\; \bigl\{\, \mathbf{x}_t \,\bigr\}_{t=0}^{\,n-1},
\qquad
\mathbf{x}_t \in \mathbb{R}^{40 \times 2}
$$

То есть каждая буква — это **растянутая во времени последовательность** «снимков» формы губ:

$$
\mathbf{x}_0 \;\longrightarrow\; \mathbf{x}_1 \;\longrightarrow\; \cdots \;\longrightarrow\; \mathbf{x}_{n-1}
$$

где каждый $\mathbf{x}_t$ — плоская конфигурация из 40 точек:

$$
\mathbf{x}_t \;=\;
\bigl\{\, (x_p,\, y_p) \,\bigr\}_{p=0}^{39}
$$


### 🎬 Визуализация: функция `animate_lips`

Преподаватель предоставил **готовую функцию** для интерактивной визуализации:

```python
animate_lips(sequence, interval=...)
```
Предлагается оставить значение интервала равным 200. Однако во время интерполяции желательно увеличивать это время пропорционально отнощению желанного количества отрисованных кадров к исходному количеству кадров
### Ниже представлен пример для буквы А

In [37]:
animate_lips(LipsData['Ж'], interval=200)

## 📐 Часть 1. Интерполяционный многочлен Лагранжа


#### НАПОМНИМ

Пусть даны $n$ узлов $t_0 < t_1 < \dots < t_{n-1}$ и значения функции в них $y_0, y_1, \dots, y_{n-1}$.

**Задача:** найти многочлен $P(t)$ степени $\le n-1$, который **точно проходит через все узлы**:

$$
P(t_i) = y_i, \qquad i = 0, \dots, n-1
$$

---

#### 🧮 Формула Лагранжа

Решение единственно и записывается в виде

$$
P(t) \;=\; \sum_{i=0}^{n-1} y_i \, L_i(t)
$$

где **базисные многочлены Лагранжа**:

$$
L_i(t) \;=\; \prod_{\substack{j=0 \\ j \ne i}}^{n-1} \frac{t - t_j}{\,t_i - t_j\,}
$$

---

#### ✨ Ключевое свойство

$$
L_i(t_k) \;=\;
\begin{cases}
1, & i = k \\[4pt]
0, & i \ne k
\end{cases}
$$

То есть каждый базисный многочлен «отвечает» **только за свой узел**.  
Отсюда мгновенно следует:

$$
P(t_k) = \sum_i y_i \, L_i(t_k) = y_k \quad ✔
$$

---



#### 🤔 Что же делать в части 1 ???

Мы применим эту формулу к **каждой из 80 координат** наших губ:

$$
(x_0, y_0),\;(x_1, y_1),\;\dots,\;(x_{39}, y_{39})
$$

и восстановим **пропущенные кадры** между известными узлами.

#########################################################################################################################

### 🎯 Задание

Дан словарь `LipsData`, где:

- 🔑 ключ — буква,
- 📊 значение — массив `(n, 40, 2)`.

**Требуется:**

1. Для каждой буквы $\ell$ построить **интерполяционный многочлен Лагранжа**
   для каждой из $40 \times 2 = 80$ координат.

   
   В качестве узлов использовать **все исходные кадры**:

   $$
   t_i = 0,\; 1,\; 2,\; \dots,\; n-1
   $$

3. Вычислить значения многочлена на **вдвое более плотной сетке**:

   $$
   t = 0,\; 0.5,\; 1,\; 1.5,\; 2,\; 2.5,\; \dots,\; n-1
   $$

   То есть кадров станет **в 2 раза больше**, чем было изначально.

4. Собрать результат в массив формы `(2n − 1, 40, 2)`.

5. **Визуализировать** результат функцией `animate_lips`.

---

#### ✅ Что должно получиться

- 📈 Каждая буква — восстановленная последовательность `(n, 40, 2)`.
- 🎬 Плавная анимация, где промежуточные кадры заполнены.



In [ ]:
import numpy as np

def lagrange_interpolation_vectorized(t_old, y_nodes, t_new):
    n = len(t_old)
    result = np.zeros(len(t_new))
    for i in range(n):
        L = np.ones(len(t_new))
        for j in range(n):
            if i != j:
                L *= (t_new - t_old[j]) / (t_old[i] - t_old[j])
        result += y_nodes[i] * L
    return result


def interpolate_lips(lip_data):
    n_frames = lip_data.shape[0]
    n_points = lip_data.shape[1]
    t_old = np.arange(n_frames, dtype=float)
    t_new = np.arange(0, n_frames - 0.5, 0.5)
    
    result = np.zeros((len(t_new), n_points, 2), dtype=float)
    for p in range(n_points):
        result[:, p, 0] = lagrange_interpolation_vectorized(t_old, lip_data[:, p, 0], t_new)
        result[:, p, 1] = lagrange_interpolation_vectorized(t_old, lip_data[:, p, 1], t_new)
    return result


def draw_frame(lip_sequence, frame_index, total_frames, title=None):
    fig, ax = plt.subplots(figsize=(10, 8))
    points = lip_sequence[frame_index]
    outer = points[:20]
    inner = points[20:]
    ax.plot(np.vstack([outer, outer[0]])[:, 0], np.vstack([outer, outer[0]])[:, 1], 'r-', linewidth=2)
    ax.plot(outer[:, 0], outer[:, 1], 'ro', markersize=6)
    ax.plot(np.vstack([inner, inner[0]])[:, 0], np.vstack([inner, inner[0]])[:, 1], 'r-', linewidth=2)
    ax.plot(inner[:, 0], inner[:, 1], 'ro', markersize=6)
    ax.set_title(title or f'Кадр {frame_index + 1}/{total_frames}', fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()


TaskListPoints = {}
for letter, lip_data in LipsData.items():
    print(f"Буква {letter}: {lip_data.shape[0]} → {2 * lip_data.shape[0] - 1} кадров")
    TaskListPoints[letter] = interpolate_lips(lip_data)

for letter, data in TaskListPoints.items():
    print(f"{letter}: {data.shape}")

data = TaskListPoints['Ж']
n = data.shape[0]
animate_lips(data, interval=400)



Буква Ё: 7 → 13 кадров
Буква А: 10 → 19 кадров
Буква Б: 12 → 23 кадров
Буква В: 8 → 15 кадров
Буква Г: 7 → 13 кадров
Буква Д: 7 → 13 кадров
Буква Е: 7 → 13 кадров
Буква Ж: 6 → 11 кадров
Буква З: 6 → 11 кадров
Ё: (13, 40, 2)
А: (19, 40, 2)
Б: (23, 40, 2)
В: (15, 40, 2)
Г: (13, 40, 2)
Д: (13, 40, 2)
Е: (13, 40, 2)
Ж: (11, 40, 2)
З: (11, 40, 2)


### ⚠️ Внимание!

> **Не ждите, что результаты будут прям красивыми** 😅  


> 🧠 **Если что-то не получается — это нормально!**  
> Подумайте, **почему** результат выглядит именно так,  
> и выпишите свои мысли в ячейке ниже. ✍️

#### ✍️ Запишите здесь, что вы наблюдаете и почему, по вашему мнению,
####    результат выглядит именно так.
\#
    ВАШИ МЫСЛИ
\#

In [ ]:
#########################################################################################################################

## 🌊 Часть 2. Кубические сплайны

Многочлен Лагранжа на большом числе узлов **нестабилен** — феномен Рунге делает своё дело 🔥

**Сплайны** решают эту проблему: вместо одного «здоровенного» многочлена мы строим **много маленьких кусочков**, которые аккуратно сшиваются между собой.

---

### 📐 Кубический сплайн дефекта 1

Дан отрезок $[a, b]$ и узлы $t_0 < t_1 < \dots < t_n$.  
Кубический сплайн $S(t)$ — это функция, которая:

1. На каждом подынтервале $[t_i, t_{i+1}]$ является **кубическим многочленом**:
   $$
   S(t) = a_i + b_i (t - t_i) + c_i (t - t_i)^2 + d_i (t - t_i)^3
   $$

2. **Проходит через все узлы**: $S(t_i) = y_i$

3. **Непрерывна** вместе с первой и второй производной:
   $$
   S, \; S', \; S'' \;\text{— непрерывны на всём } [a, b]
   $$

4. На концах задаётся **краевое условие** (например, $S''(a) = S''(b) = 0$ — *natural spline*).

> 💡 **Дефект 1** означает, что «склейка» идеальная: разрывается только третья производная, а $S$, $S'$, $S''$ — гладкие.

---

### 🧩 Задание 2.1 — заменить Лагранжа сплайнами

1. Реализуйте метод построения **кубического сплайна дефекта 1** из лекции.
2. Примените его к той же задаче, что и в Части 1:
   - узлы — все исходные такты $t_i = 0, 1, 2, \dots, n-1$,
   - значения — координаты губ,
   - восстановление на сетке с шагом $0.5$.



Проверка:

```python
animate_lips(SplineLipsLetters['А'], interval=100)
```

> 🧠 **Сравните** с результатом Лагранжа.  
> Запишите в ячейке ниже, **чем сплайны отличаются** от многочлена Лагранжа визуально.

---

## 🔗 Задание 2.2 — склеиваем звуки в слова

Теперь попробуем **соединять звуки** между собой — как в реальной речи, где один звук плавно перетекает в другой.

### 🛠️ Подготовка

1. **Вручную просмотрите** каждый звук через `animate_lips`.
2. **Обрежьте** крайние кадры, где рот закрывается или стоит без движения.  
   💡 *Можно использовать уже восстановленные кадры из задания 2.1 — они глаже.*
3. Оставьте только «рабочую» часть, где **губы действительно меняются**.

### 🧪 Эксперимент

Соберите **связки из 2–3 букв**, например:

```python
'АБ',  'БА',  'АБВ',  'ОБА',  ...
```

Для каждой связки:

- 🔴 **сначала** соедините куски **многочленом Лагранжа**,
- 🔵 **потом** — **кубическим сплайном**.
#### *Это независимые действия, то есть должно быть два результата: звук получен Лагранжем, а потом Сплайном*

Сравните анимации.

### 🎬 Финальный тест

Соберите слово:

```
абёба
```

и покажите его анимацию — сначала Лагранжем, потом сплайном.

---

## ✍️ Что записать в отчёте

Опишите **своими словами**:

- 🔍 Что происходит на **стыках** звуков у Лагранжа? А у сплайнов?
- 📉 Есть ли **разрывы**, **скачки**, **выбросы**?
- 👀 Какой метод выглядит **естественнее**?
- 🤔 Почему?

---

> 💡 **Подсказка:** чтобы соединить два звука, просто **склейте их массивы по оси времени**:
> ```python
> combined = np.concatenate([part1, part2], axis=0)
> ```
> А затем **интерполируйте уже склейку** — там будет видно, справляется ли метод с «переходом».

<div align="center">

# 🚧 COMING SOON 🚧

## 📉 Часть 3. Метод наименьших квадратов (МНК)

Всё, что мы делали до сих пор, было **интерполяцией** — многочлен **обязан** пройти через каждую точку.  
Именно это и приводит к **феномену Рунге**: пытаясь угодить всем узлам сразу, многочлен «сходит с ума» на краях.

**Метод наименьших квадратов** поступает иначе: мы больше **не требуем** точного прохождения через точки.  
Вместо этого ищем функцию, которая **в среднем** проходит как можно ближе ко всем данным — и это делает её куда **устойчивее** и **гладче**.

🎯 Мы попробуем сгладить наши губы МНК и сравнить результат с тем, что уже получили.


> #### 🚨 ВНИМАНИЕ! 🚨
>
> ##### 📉 Часть 3 (МНК) — последняя ОБЯЗАТЕЛЬНАЯ ЗАДАЧА для сдачи (и, естественно, допуска к экзамену)
>
> ✅ Дальше — только по желанию 🌟


## 🌟 Часть 4**** (необязательная). МНК + PCA

До сих пор мы обрабатывали **каждую из 80 координат независимо**.  
Но губы — это **связанная система**: точки двигаются согласованно, а не поодиночке.

**Метод главных компонент (PCA)** находит **опорные векторы** — направления, вдоль которых форма губ меняется сильнее всего.  
И тогда вместо 80 координат мы работаем с **несколькими коэффициентами** в новом базисе.

> 🎯 Интерполяция будет вестись уже **в пространстве главных компонент** —  
> по опорным векторам, которые описывают саму «суть» движения губ.  
> Это самый **естественный** и **устойчивый** подход из всех, что мы попробуем.

# Практическое долгосрочное задание 1
## 👄 Интерполяция формы губ по видео



## ЧАСТЬ 0. 🎯 Введение

Нам в руки попали **обрывки данных** — редкие снимки губ человека, произносящего звуки **восточноевропейского алфавита** 🕵️

Наш агент, внедрённый глубоко под прикрытием, снимал его **раз в секунду** — строго по **равномерной временно́й сетке**. Никаких отклонений, никаких дополнительных замеров. Только сухие факты: кадр, ещё кадр, и снова тишина.




Между этими кадрами — **пустота**. Звуки шли один за другим, но мы видим лишь **редкие вспышки мимики**: губы то округляются, то растягиваются, то сжимаются — и всё это **урывками**, без плавных переходов. Мимика кажется **рваной**, движения — **дёргаными**. Смотреть на такое больно 😖

---

### 🚀 Наша финальная задача

#### Научиться воспроизводить **гладкую анимацию** для… 🌀

…впрочем, узнаем **потом**. 😏

---

### 🎬 А пока — начнём с малого!

#### Попробуем восстановить хотя бы **отдельные звуки** 🗣️

<table>
<tr>
<td align="center">👄</td>
<td align="center">🎞️</td>
<td align="center">✨</td>
<td align="center">🔊</td>
</tr>
<tr>
<td align="center">губы</td>
<td align="center">кадры</td>
<td align="center">сгладим</td>
<td align="center">зазвучит!</td>
</tr>
</table>

---

> 💭 **Шаг за шагом:**
> 1. 📍 Соберём редкие кадры
> 2. 🧮 Построим интерполяцию
> 3. 🎥 Оживим анимацию
> 4. 🌍 А потом — уже что-то большее…

In [ ]:
#Импорт нужных библиотек

from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy
import glob
from collections import defaultdict
from draw import *

import pickle

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
#Загрузка данных
with open('LipsArrayHalfTime.np' , 'rb') as f:
    LipsData = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'LipsArrayHalfTime.np'

### 📦 Структура данных: `LipsData`

`LipsData` — это **словарь**, ставящий в соответствие каждой букве её временну́ю траекторию движения губ:

$$
\text{LipsData} : \quad \text{буква} \;\longmapsto\; \mathbf{X} \in \mathbb{R}^{n \times 40 \times 2}
$$

---

#### 📐 Что означает каждый индекс

$$
\mathbf{X}[t,\, p,\, c], \qquad
\begin{aligned}
&t = 0, \dots, n-1 \\
&p = 0, \dots, 39 \\
&c \in \{0, 1\}
\end{aligned}
$$

| Индекс | Обозначение | Смысл |
|:---:|:---:|:---|
| $t$ | **время** | номер такта — дискретный отсчёт вдоль оси времени |
| $p$ | **точка** | 40 ключевых точек контура губ (20 внешних + 20 внутренних) |
| $c$ | **координата** | $c=0 \Rightarrow x$, $\;c=1 \Rightarrow y$ на плоскости |

---

#### 🕰️ О такте

> **Такт** — это условная единица времени.  
> Можно считать, что $1$ такт $\approx 0.1$ с, но эта привязка несущественна:  
> для интерполяции важна лишь **последовательность** отсчётов, а не их физическая длительность.

---

#### 🧠 Как это читать

Для фиксированной буквы $\ell$ имеем:

$$
\text{LipsData}[\ell] \;=\; \bigl\{\, \mathbf{x}_t \,\bigr\}_{t=0}^{\,n-1},
\qquad
\mathbf{x}_t \in \mathbb{R}^{40 \times 2}
$$

То есть каждая буква — это **растянутая во времени последовательность** «снимков» формы губ:

$$
\mathbf{x}_0 \;\longrightarrow\; \mathbf{x}_1 \;\longrightarrow\; \cdots \;\longrightarrow\; \mathbf{x}_{n-1}
$$

где каждый $\mathbf{x}_t$ — плоская конфигурация из 40 точек:

$$
\mathbf{x}_t \;=\;
\bigl\{\, (x_p,\, y_p) \,\bigr\}_{p=0}^{39}
$$


### 🎬 Визуализация: функция `animate_lips`

Преподаватель предоставил **готовую функцию** для интерактивной визуализации:

```python
animate_lips(sequence, interval=...)
```
Предлагается оставить значение интервала равным 200. Однако во время интерполяции желательно увеличивать это время пропорционально отнощению желанного количества отрисованных кадров к исходному количеству кадров
### Ниже представлен пример для буквы А

In [ ]:
animate_lips(LipsData['А'], interval=200)

## 📐 Часть 1. Интерполяционный многочлен Лагранжа


#### НАПОМНИМ

Пусть даны $n$ узлов $t_0 < t_1 < \dots < t_{n-1}$ и значения функции в них $y_0, y_1, \dots, y_{n-1}$.

**Задача:** найти многочлен $P(t)$ степени $\le n-1$, который **точно проходит через все узлы**:

$$
P(t_i) = y_i, \qquad i = 0, \dots, n-1
$$

---

#### 🧮 Формула Лагранжа

Решение единственно и записывается в виде

$$
P(t) \;=\; \sum_{i=0}^{n-1} y_i \, L_i(t)
$$

где **базисные многочлены Лагранжа**:

$$
L_i(t) \;=\; \prod_{\substack{j=0 \\ j \ne i}}^{n-1} \frac{t - t_j}{\,t_i - t_j\,}
$$

---

#### ✨ Ключевое свойство

$$
L_i(t_k) \;=\;
\begin{cases}
1, & i = k \\[4pt]
0, & i \ne k
\end{cases}
$$

То есть каждый базисный многочлен «отвечает» **только за свой узел**.  
Отсюда мгновенно следует:

$$
P(t_k) = \sum_i y_i \, L_i(t_k) = y_k \quad ✔
$$

---



#### 🤔 Что же делать в части 1 ???

Мы применим эту формулу к **каждой из 80 координат** наших губ:

$$
(x_0, y_0),\;(x_1, y_1),\;\dots,\;(x_{39}, y_{39})
$$

и восстановим **пропущенные кадры** между известными узлами.

#########################################################################################################################

### 🎯 Задание

Дан словарь `LipsData`, где:

- 🔑 ключ — буква,
- 📊 значение — массив `(n, 40, 2)`.

**Требуется:**

1. Для каждой буквы $\ell$ построить **интерполяционный многочлен Лагранжа**
   для каждой из $40 \times 2 = 80$ координат.

   
   В качестве узлов использовать **все исходные кадры**:

   $$
   t_i = 0,\; 1,\; 2,\; \dots,\; n-1
   $$

3. Вычислить значения многочлена на **вдвое более плотной сетке**:

   $$
   t = 0,\; 0.5,\; 1,\; 1.5,\; 2,\; 2.5,\; \dots,\; n-1
   $$

   То есть кадров станет **в 2 раза больше**, чем было изначально.

4. Собрать результат в массив формы `(2n − 1, 40, 2)`.

5. **Визуализировать** результат функцией `animate_lips`.

---

#### ✅ Что должно получиться

- 📈 Каждая буква — восстановленная последовательность `(n, 40, 2)`.
- 🎬 Плавная анимация, где промежуточные кадры заполнены.



In [ ]:
import numpy as np

def lagrange_interpolation(x_nodes, y_nodes, x):
    """
    Интерполяция Лагранжа в точке x.
    x_nodes — исходные узлы
    y_nodes — значения функции в узлах
    """

    result = 0.0
    n = len(x_nodes)

    for i in range(n):
        L = 1.0

        for j in range(n):
            if i != j:
                L *= (x - x_nodes[j]) / (x_nodes[i] - x_nodes[j])

        result += y_nodes[i] * L

    return result


def interpolate_lips(lip_data):
    n_frames = lip_data.shape[0]
    n_points = lip_data.shape[1]


    t_old = np.arange(n_frames, dtype=float)
    t_new = np.arange(0, n_frames - 1 + 0.5, 0.5)

    result = np.zeros(
        (len(t_new), n_points, 2),
        dtype=float
    )

    for p in range(n_points):
        x_values = lip_data[:, p, 0]
        y_values = lip_data[:, p, 1]
        for k, t in enumerate(t_new):

            result[k, p, 0] = lagrange_interpolation(
                t_old,
                x_values,
                t
            )

            result[k, p, 1] = lagrange_interpolation(
                t_old,
                y_values,
                t
            )

    return result

TaskListPoints = {}

for letter, lip_data in LipsData.items():

    print(
        f"Буква {letter}: "
        f"{lip_data.shape[0]} → {2 * lip_data.shape[0] - 1} кадров"
    )

    TaskListPoints[letter] = interpolate_lips(lip_data)

for letter, data in TaskListPoints.items():

    print(
        f"{letter}: "
        f"{data.shape}"
    )

ModuleNotFoundError: No module named 'numpy'

### ⚠️ Внимание!

> **Не ждите, что результаты будут прям красивыми** 😅  


> 🧠 **Если что-то не получается — это нормально!**  
> Подумайте, **почему** результат выглядит именно так,  
> и выпишите свои мысли в ячейке ниже. ✍️

#### ✍️ Запишите здесь, что вы наблюдаете и почему, по вашему мнению,
####    результат выглядит именно так.
\#
    ВАШИ МЫСЛИ
\#

In [ ]:
#########################################################################################################################

## 🌊 Часть 2. Кубические сплайны

Многочлен Лагранжа на большом числе узлов **нестабилен** — феномен Рунге делает своё дело 🔥

**Сплайны** решают эту проблему: вместо одного «здоровенного» многочлена мы строим **много маленьких кусочков**, которые аккуратно сшиваются между собой.

---

### 📐 Кубический сплайн дефекта 1

Дан отрезок $[a, b]$ и узлы $t_0 < t_1 < \dots < t_n$.  
Кубический сплайн $S(t)$ — это функция, которая:

1. На каждом подынтервале $[t_i, t_{i+1}]$ является **кубическим многочленом**:
   $$
   S(t) = a_i + b_i (t - t_i) + c_i (t - t_i)^2 + d_i (t - t_i)^3
   $$

2. **Проходит через все узлы**: $S(t_i) = y_i$

3. **Непрерывна** вместе с первой и второй производной:
   $$
   S, \; S', \; S'' \;\text{— непрерывны на всём } [a, b]
   $$

4. На концах задаётся **краевое условие** (например, $S''(a) = S''(b) = 0$ — *natural spline*).

> 💡 **Дефект 1** означает, что «склейка» идеальная: разрывается только третья производная, а $S$, $S'$, $S''$ — гладкие.

---

### 🧩 Задание 2.1 — заменить Лагранжа сплайнами

1. Реализуйте метод построения **кубического сплайна дефекта 1** из лекции.
2. Примените его к той же задаче, что и в Части 1:
   - узлы — все исходные такты $t_i = 0, 1, 2, \dots, n-1$,
   - значения — координаты губ,
   - восстановление на сетке с шагом $0.5$.



Проверка:

```python
animate_lips(SplineLipsLetters['А'], interval=100)
```

> 🧠 **Сравните** с результатом Лагранжа.  
> Запишите в ячейке ниже, **чем сплайны отличаются** от многочлена Лагранжа визуально.

---

## 🔗 Задание 2.2 — склеиваем звуки в слова

Теперь попробуем **соединять звуки** между собой — как в реальной речи, где один звук плавно перетекает в другой.

### 🛠️ Подготовка

1. **Вручную просмотрите** каждый звук через `animate_lips`.
2. **Обрежьте** крайние кадры, где рот закрывается или стоит без движения.  
   💡 *Можно использовать уже восстановленные кадры из задания 2.1 — они глаже.*
3. Оставьте только «рабочую» часть, где **губы действительно меняются**.

### 🧪 Эксперимент

Соберите **связки из 2–3 букв**, например:

```python
'АБ',  'БА',  'АБВ',  'ОБА',  ...
```

Для каждой связки:

- 🔴 **сначала** соедините куски **многочленом Лагранжа**,
- 🔵 **потом** — **кубическим сплайном**.
#### *Это независимые действия, то есть должно быть два результата: звук получен Лагранжем, а потом Сплайном*

Сравните анимации.

### 🎬 Финальный тест

Соберите слово:

```
абёба
```

и покажите его анимацию — сначала Лагранжем, потом сплайном.

---

## ✍️ Что записать в отчёте

Опишите **своими словами**:

- 🔍 Что происходит на **стыках** звуков у Лагранжа? А у сплайнов?
- 📉 Есть ли **разрывы**, **скачки**, **выбросы**?
- 👀 Какой метод выглядит **естественнее**?
- 🤔 Почему?

---

> 💡 **Подсказка:** чтобы соединить два звука, просто **склейте их массивы по оси времени**:
> ```python
> combined = np.concatenate([part1, part2], axis=0)
> ```
> А затем **интерполируйте уже склейку** — там будет видно, справляется ли метод с «переходом».

<div align="center">

# 🚧 COMING SOON 🚧

## 📉 Часть 3. Метод наименьших квадратов (МНК)

Всё, что мы делали до сих пор, было **интерполяцией** — многочлен **обязан** пройти через каждую точку.  
Именно это и приводит к **феномену Рунге**: пытаясь угодить всем узлам сразу, многочлен «сходит с ума» на краях.

**Метод наименьших квадратов** поступает иначе: мы больше **не требуем** точного прохождения через точки.  
Вместо этого ищем функцию, которая **в среднем** проходит как можно ближе ко всем данным — и это делает её куда **устойчивее** и **гладче**.

🎯 Мы попробуем сгладить наши губы МНК и сравнить результат с тем, что уже получили.


> #### 🚨 ВНИМАНИЕ! 🚨
>
> ##### 📉 Часть 3 (МНК) — последняя ОБЯЗАТЕЛЬНАЯ ЗАДАЧА для сдачи (и, естественно, допуска к экзамену)
>
> ✅ Дальше — только по желанию 🌟


## 🌟 Часть 4**** (необязательная). МНК + PCA

До сих пор мы обрабатывали **каждую из 80 координат независимо**.  
Но губы — это **связанная система**: точки двигаются согласованно, а не поодиночке.

**Метод главных компонент (PCA)** находит **опорные векторы** — направления, вдоль которых форма губ меняется сильнее всего.  
И тогда вместо 80 координат мы работаем с **несколькими коэффициентами** в новом базисе.

> 🎯 Интерполяция будет вестись уже **в пространстве главных компонент** —  
> по опорным векторам, которые описывают саму «суть» движения губ.  
> Это самый **естественный** и **устойчивый** подход из всех, что мы попробуем.

In [5]:
#Загрузка данных
with open('LipsArrayHalfTime.np' , 'rb') as f:
    LipsData = pickle.load(f)

FileNotFoundError: [Errno 2] No such file or directory: 'LipsArrayHalfTime.np'

### 📦 Структура данных: `LipsData`

`LipsData` — это **словарь**, ставящий в соответствие каждой букве её временну́ю траекторию движения губ:

$$
\text{LipsData} : \quad \text{буква} \;\longmapsto\; \mathbf{X} \in \mathbb{R}^{n \times 40 \times 2}
$$

---

#### 📐 Что означает каждый индекс

$$
\mathbf{X}[t,\, p,\, c], \qquad
\begin{aligned}
&t = 0, \dots, n-1 \\
&p = 0, \dots, 39 \\
&c \in \{0, 1\}
\end{aligned}
$$

| Индекс | Обозначение | Смысл |
|:---:|:---:|:---|
| $t$ | **время** | номер такта — дискретный отсчёт вдоль оси времени |
| $p$ | **точка** | 40 ключевых точек контура губ (20 внешних + 20 внутренних) |
| $c$ | **координата** | $c=0 \Rightarrow x$, $\;c=1 \Rightarrow y$ на плоскости |

---

#### 🕰️ О такте

> **Такт** — это условная единица времени.  
> Можно считать, что $1$ такт $\approx 0.1$ с, но эта привязка несущественна:  
> для интерполяции важна лишь **последовательность** отсчётов, а не их физическая длительность.

---

#### 🧠 Как это читать

Для фиксированной буквы $\ell$ имеем:

$$
\text{LipsData}[\ell] \;=\; \bigl\{\, \mathbf{x}_t \,\bigr\}_{t=0}^{\,n-1},
\qquad
\mathbf{x}_t \in \mathbb{R}^{40 \times 2}
$$

То есть каждая буква — это **растянутая во времени последовательность** «снимков» формы губ:

$$
\mathbf{x}_0 \;\longrightarrow\; \mathbf{x}_1 \;\longrightarrow\; \cdots \;\longrightarrow\; \mathbf{x}_{n-1}
$$

где каждый $\mathbf{x}_t$ — плоская конфигурация из 40 точек:

$$
\mathbf{x}_t \;=\;
\bigl\{\, (x_p,\, y_p) \,\bigr\}_{p=0}^{39}
$$


### 🎬 Визуализация: функция `animate_lips`

Преподаватель предоставил **готовую функцию** для интерактивной визуализации:

```python
animate_lips(sequence, interval=...)
```
Предлагается оставить значение интервала равным 200. Однако во время интерполяции желательно увеличивать это время пропорционально отнощению желанного количества отрисованных кадров к исходному количеству кадров
### Ниже представлен пример для буквы А

In [ ]:
animate_lips(LipsData['А'], interval=200)

## 📐 Часть 1. Интерполяционный многочлен Лагранжа


#### НАПОМНИМ

Пусть даны $n$ узлов $t_0 < t_1 < \dots < t_{n-1}$ и значения функции в них $y_0, y_1, \dots, y_{n-1}$.

**Задача:** найти многочлен $P(t)$ степени $\le n-1$, который **точно проходит через все узлы**:

$$
P(t_i) = y_i, \qquad i = 0, \dots, n-1
$$

---

#### 🧮 Формула Лагранжа

Решение единственно и записывается в виде

$$
P(t) \;=\; \sum_{i=0}^{n-1} y_i \, L_i(t)
$$

где **базисные многочлены Лагранжа**:

$$
L_i(t) \;=\; \prod_{\substack{j=0 \\ j \ne i}}^{n-1} \frac{t - t_j}{\,t_i - t_j\,}
$$

---

#### ✨ Ключевое свойство

$$
L_i(t_k) \;=\;
\begin{cases}
1, & i = k \\[4pt]
0, & i \ne k
\end{cases}
$$

То есть каждый базисный многочлен «отвечает» **только за свой узел**.  
Отсюда мгновенно следует:

$$
P(t_k) = \sum_i y_i \, L_i(t_k) = y_k \quad ✔
$$

---



#### 🤔 Что же делать в части 1 ???

Мы применим эту формулу к **каждой из 80 координат** наших губ:

$$
(x_0, y_0),\;(x_1, y_1),\;\dots,\;(x_{39}, y_{39})
$$

и восстановим **пропущенные кадры** между известными узлами.

#########################################################################################################################

### 🎯 Задание

Дан словарь `LipsData`, где:

- 🔑 ключ — буква,
- 📊 значение — массив `(n, 40, 2)`.

**Требуется:**

1. Для каждой буквы $\ell$ построить **интерполяционный многочлен Лагранжа**
   для каждой из $40 \times 2 = 80$ координат.

   
   В качестве узлов использовать **все исходные кадры**:

   $$
   t_i = 0,\; 1,\; 2,\; \dots,\; n-1
   $$

3. Вычислить значения многочлена на **вдвое более плотной сетке**:

   $$
   t = 0,\; 0.5,\; 1,\; 1.5,\; 2,\; 2.5,\; \dots,\; n-1
   $$

   То есть кадров станет **в 2 раза больше**, чем было изначально.

4. Собрать результат в массив формы `(2n − 1, 40, 2)`.

5. **Визуализировать** результат функцией `animate_lips`.

---

#### ✅ Что должно получиться

- 📈 Каждая буква — восстановленная последовательность `(n, 40, 2)`.
- 🎬 Плавная анимация, где промежуточные кадры заполнены.



In [8]:
import numpy as np

def lagrange_interpolation(x_nodes, y_nodes, x):
    """
    Интерполяция Лагранжа в точке x.
    x_nodes — исходные узлы
    y_nodes — значения функции в узлах
    """

    result = 0.0
    n = len(x_nodes)

    for i in range(n):
        L = 1.0

        for j in range(n):
            if i != j:
                L *= (x - x_nodes[j]) / (x_nodes[i] - x_nodes[j])

        result += y_nodes[i] * L

    return result


def interpolate_lips(lip_data):
    n_frames = lip_data.shape[0]
    n_points = lip_data.shape[1]


    t_old = np.arange(n_frames, dtype=float)
    t_new = np.arange(0, n_frames - 1 + 0.5, 0.5)

    result = np.zeros(
        (len(t_new), n_points, 2),
        dtype=float
    )

    for p in range(n_points):
        x_values = lip_data[:, p, 0]
        y_values = lip_data[:, p, 1]
        for k, t in enumerate(t_new):

            result[k, p, 0] = lagrange_interpolation(
                t_old,
                x_values,
                t
            )

            result[k, p, 1] = lagrange_interpolation(
                t_old,
                y_values,
                t
            )

    return result

TaskListPoints = {}

for letter, lip_data in LipsData.items():

    print(
        f"Буква {letter}: "
        f"{lip_data.shape[0]} → {2 * lip_data.shape[0] - 1} кадров"
    )

    TaskListPoints[letter] = interpolate_lips(lip_data)

for letter, data in TaskListPoints.items():

    print(
        f"{letter}: "
        f"{data.shape}"
    )

ModuleNotFoundError: No module named 'numpy'

### ⚠️ Внимание!

> **Не ждите, что результаты будут прям красивыми** 😅  


> 🧠 **Если что-то не получается — это нормально!**  
> Подумайте, **почему** результат выглядит именно так,  
> и выпишите свои мысли в ячейке ниже. ✍️

#### ✍️ Запишите здесь, что вы наблюдаете и почему, по вашему мнению,
####    результат выглядит именно так.
\#
    ВАШИ МЫСЛИ
\#

In [ ]:
#########################################################################################################################

## 🌊 Часть 2. Кубические сплайны

Многочлен Лагранжа на большом числе узлов **нестабилен** — феномен Рунге делает своё дело 🔥

**Сплайны** решают эту проблему: вместо одного «здоровенного» многочлена мы строим **много маленьких кусочков**, которые аккуратно сшиваются между собой.

---

### 📐 Кубический сплайн дефекта 1

Дан отрезок $[a, b]$ и узлы $t_0 < t_1 < \dots < t_n$.  
Кубический сплайн $S(t)$ — это функция, которая:

1. На каждом подынтервале $[t_i, t_{i+1}]$ является **кубическим многочленом**:
   $$
   S(t) = a_i + b_i (t - t_i) + c_i (t - t_i)^2 + d_i (t - t_i)^3
   $$

2. **Проходит через все узлы**: $S(t_i) = y_i$

3. **Непрерывна** вместе с первой и второй производной:
   $$
   S, \; S', \; S'' \;\text{— непрерывны на всём } [a, b]
   $$

4. На концах задаётся **краевое условие** (например, $S''(a) = S''(b) = 0$ — *natural spline*).

> 💡 **Дефект 1** означает, что «склейка» идеальная: разрывается только третья производная, а $S$, $S'$, $S''$ — гладкие.

---

### 🧩 Задание 2.1 — заменить Лагранжа сплайнами

1. Реализуйте метод построения **кубического сплайна дефекта 1** из лекции.
2. Примените его к той же задаче, что и в Части 1:
   - узлы — все исходные такты $t_i = 0, 1, 2, \dots, n-1$,
   - значения — координаты губ,
   - восстановление на сетке с шагом $0.5$.



Проверка:

```python
animate_lips(SplineLipsLetters['А'], interval=100)
```

> 🧠 **Сравните** с результатом Лагранжа.  
> Запишите в ячейке ниже, **чем сплайны отличаются** от многочлена Лагранжа визуально.

---

## 🔗 Задание 2.2 — склеиваем звуки в слова

Теперь попробуем **соединять звуки** между собой — как в реальной речи, где один звук плавно перетекает в другой.

### 🛠️ Подготовка

1. **Вручную просмотрите** каждый звук через `animate_lips`.
2. **Обрежьте** крайние кадры, где рот закрывается или стоит без движения.  
   💡 *Можно использовать уже восстановленные кадры из задания 2.1 — они глаже.*
3. Оставьте только «рабочую» часть, где **губы действительно меняются**.

### 🧪 Эксперимент

Соберите **связки из 2–3 букв**, например:

```python
'АБ',  'БА',  'АБВ',  'ОБА',  ...
```

Для каждой связки:

- 🔴 **сначала** соедините куски **многочленом Лагранжа**,
- 🔵 **потом** — **кубическим сплайном**.
#### *Это независимые действия, то есть должно быть два результата: звук получен Лагранжем, а потом Сплайном*

Сравните анимации.

### 🎬 Финальный тест

Соберите слово:

```
абёба
```

и покажите его анимацию — сначала Лагранжем, потом сплайном.

---

## ✍️ Что записать в отчёте

Опишите **своими словами**:

- 🔍 Что происходит на **стыках** звуков у Лагранжа? А у сплайнов?
- 📉 Есть ли **разрывы**, **скачки**, **выбросы**?
- 👀 Какой метод выглядит **естественнее**?
- 🤔 Почему?

---

> 💡 **Подсказка:** чтобы соединить два звука, просто **склейте их массивы по оси времени**:
> ```python
> combined = np.concatenate([part1, part2], axis=0)
> ```
> А затем **интерполируйте уже склейку** — там будет видно, справляется ли метод с «переходом».

<div align="center">

# 🚧 COMING SOON 🚧

## 📉 Часть 3. Метод наименьших квадратов (МНК)

Всё, что мы делали до сих пор, было **интерполяцией** — многочлен **обязан** пройти через каждую точку.  
Именно это и приводит к **феномену Рунге**: пытаясь угодить всем узлам сразу, многочлен «сходит с ума» на краях.

**Метод наименьших квадратов** поступает иначе: мы больше **не требуем** точного прохождения через точки.  
Вместо этого ищем функцию, которая **в среднем** проходит как можно ближе ко всем данным — и это делает её куда **устойчивее** и **гладче**.

🎯 Мы попробуем сгладить наши губы МНК и сравнить результат с тем, что уже получили.


> #### 🚨 ВНИМАНИЕ! 🚨
>
> ##### 📉 Часть 3 (МНК) — последняя ОБЯЗАТЕЛЬНАЯ ЗАДАЧА для сдачи (и, естественно, допуска к экзамену)
>
> ✅ Дальше — только по желанию 🌟


## 🌟 Часть 4**** (необязательная). МНК + PCA

До сих пор мы обрабатывали **каждую из 80 координат независимо**.  
Но губы — это **связанная система**: точки двигаются согласованно, а не поодиночке.

**Метод главных компонент (PCA)** находит **опорные векторы** — направления, вдоль которых форма губ меняется сильнее всего.  
И тогда вместо 80 координат мы работаем с **несколькими коэффициентами** в новом базисе.

> 🎯 Интерполяция будет вестись уже **в пространстве главных компонент** —  
> по опорным векторам, которые описывают саму «суть» движения губ.  
> Это самый **естественный** и **устойчивый** подход из всех, что мы попробуем.